In [ ]:
# Standard library imports
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import altair as alt

# Statistics and ML preprocessing
from scipy.stats import pearsonr, gaussian_kde, skew, kurtosis
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer, QuantileTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split

# Set up plotting parameters
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.dpi'] = 300

# Custom color palette - organized by families
colors = {
    'purple': ['#962955', '#8c397b', '#b684d5', '#eb85c0'],
    'green': ['#7fb775', '#9ab25d', '#33510c', '#006c48', '#008a77', '#72b797', '#25522c'],
    'blue': ['#6479cc', '#26b1fd', '#48b7cd', '#0181a1', '#89afba'],
    'orange': ['#ad933c', '#a28a33', '#be6940', '#723916', '#d59e67'],
    'red': ['#d3616e', '#e77b6d']
}

# Set primary color palette
primary_colors = [colors['purple'][0], colors['green'][0], colors['blue'][0], colors['orange'][0]]
sns.set_palette(primary_colors)

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Machine Learning Data Preparation for Gene Feature Analysis

## Purpose
This notebook prepares gene feature data for machine learning models predicting DIT-HAP (Deletion-Insertion-Transformation Haploid Analysis Pipeline) metrics. We will:

1. **Load and explore** gene features and target variables
2. **Analyze data quality** through missing value patterns and distributions
3. **Visualize feature distributions** with histograms and statistical summaries
4. **Handle missing values** using appropriate imputation strategies
5. **Encode categorical variables** and scale numerical features
6. **Detect and handle outliers** that could impact model performance
7. **Prepare final datasets** for machine learning workflows

## Data Sources
- **Gene features**: Comprehensive molecular and genomic characteristics
- **DIT-HAP metrics**: Curve fitting parameters (A, um, lam) and cluster assignments
- **Target variables**: Growth curve parameters and cluster classifications

---

# 1. Data Loading and Initial Exploration

In [ ]:
# Define output directory
output_dir = Path("../../results/HD_DIT_HAP_generationRAW/22_machine_learning_modeling/")
output_dir.mkdir(parents=True, exist_ok=True)

print("Loading datasets...")

# Load gene features
try:
    gene_features = pd.read_csv("../../resources/pombe_features/pombe_coding_gene_protein_features.tsv", sep="\t")
    print(f"✓ Gene features loaded: {gene_features.shape}")
except FileNotFoundError:
    print("✗ Gene features file not found")
    raise

# Load DIT-HAP gene level metrics
try:
    DIT_HAP_gene_level_metrics = pd.read_csv(
        "../../results/HD_DIT_HAP_generationRAW/18_gene_level_clustering/kmeans_cluster_result.tsv", 
        sep="\t"
    ).rename(columns={"Systematic ID": "Systematic_ID"})
    print(f"✓ DIT-HAP metrics loaded: {DIT_HAP_gene_level_metrics.shape}")
except FileNotFoundError:
    print("✗ DIT-HAP metrics file not found")
    raise

print("\nDataset loading completed successfully!")

In [ ]:
# Examine dataset structures and key statistics
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

datasets = {
    "Gene Features": gene_features,
    "DIT-HAP Metrics": DIT_HAP_gene_level_metrics
}

for name, df in datasets.items():
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns[:5])}{'...' if len(df.columns) > 5 else ''}")
    print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Check for duplicates
    n_duplicates = df.duplicated().sum()
    if n_duplicates > 0:
        print(f"  ⚠ Duplicate rows: {n_duplicates}")
    else:
        print(f"  ✓ No duplicate rows")

# Display sample data from each dataset
print("\n" + "=" * 60)
print("SAMPLE DATA PREVIEW")
print("=" * 60)

for name, df in datasets.items():
    print(f"\n{name} - First 3 rows:")
    display(df.head(3))


# 2. Data Integration and Target Preparation

We'll merge the datasets and examine the target variables for our machine learning models.

In [ ]:
# Merge DIT-HAP metrics with cluster assignments
print("Merging target variables...")

# Extract relevant columns for targets
target_columns = ["Systematic_ID", "A", "um", "lam", "revised_cluster"]
target_values = DIT_HAP_gene_level_metrics[target_columns].rename(columns={"revised_cluster": "DIT_HAP_cluster"})

print(f"Target values shape: {target_values.shape}")
print(f"Genes with curve fitting parameters: {target_values['A'].notna().sum()}")
print(f"Genes with cluster assignments: {target_values['DIT_HAP_cluster'].notna().sum()}")

# Examine target variable distributions
print("\nTarget Variable Summary:")
print(target_values[['A', 'um', 'lam']].describe())

# Check for any systematic ID overlaps
print(f"\nUnique genes in target data: {target_values['Systematic_ID'].nunique()}")
print(f"Total rows in target data: {len(target_values)}")

# Display sample of target values
print("\nSample target values:")
display(target_values.head())

# 3. Complete Dataset Integration

Now we'll merge all features with target values and perform comprehensive data quality assessment.

In [ ]:
# Merge gene features with target values
print("Integrating features with target values...")

all_features_with_target_values = pd.merge(
    gene_features, 
    target_values, 
    left_on="gene_systematic_id", 
    right_on="Systematic_ID",
    how="left"
).drop(columns=["Systematic_ID"]).rename(columns={"gene_systematic_id": "Systematic_ID"})
all_features_with_target_values.to_csv(output_dir / "all_features_with_target_values.csv", index=False, float_format="%.5f")

print(f"Integrated dataset shape: {all_features_with_target_values.shape}")
print(f"Genes with complete feature sets: {len(gene_features)}")
print(f"Genes with target values: {target_values['A'].notna().sum()}")
print(f"Final dataset size: {len(all_features_with_target_values)}")

# Examine the merge quality
merge_stats = {
    'Total genes': len(all_features_with_target_values),
    'Genes with A parameter': all_features_with_target_values['A'].notna().sum(),
    'Genes with um parameter': all_features_with_target_values['um'].notna().sum(),
    'Genes with lam parameter': all_features_with_target_values['lam'].notna().sum(),
    'Genes with cluster assignment': all_features_with_target_values['DIT_HAP_cluster'].notna().sum(),
    'Genes with all targets': len(all_features_with_target_values.dropna(subset=['A', 'um', 'lam', 'DIT_HAP_cluster']))
}

print("\nMerge Quality Assessment:")
for key, value in merge_stats.items():
    percentage = (value / len(all_features_with_target_values)) * 100
    print(f"  {key}: {value:,} ({percentage:.1f}%)")

print(f"\nDataset columns: {len(all_features_with_target_values.columns)}")
print(f"Feature columns: {len([col for col in all_features_with_target_values.columns if col not in ['A', 'um', 'lam', 'DIT_HAP_cluster']])}")

In [ ]:
# Comprehensive missing value analysis
print("=" * 60)
print("MISSING VALUE ANALYSIS")
print("=" * 60)

missing_values = all_features_with_target_values.isna().sum()
missing_percent = (missing_values / len(all_features_with_target_values)) * 100

# Create missing value summary
missing_summary = pd.DataFrame({
    'Missing_Count': missing_values,
    'Missing_Percent': missing_percent
}).sort_values('Missing_Percent', ascending=False)

# Filter to show only columns with missing values
missing_summary_filtered = missing_summary[missing_summary['Missing_Count'] > 0]

print(f"Columns with missing values: {len(missing_summary_filtered)}")
print(f"Columns without missing values: {len(missing_summary) - len(missing_summary_filtered)}")

print("\nTop 15 columns with highest missing values:")
display(missing_summary_filtered.head(15))

# Save missing value summary
missing_summary.to_csv(output_dir / "missing_value_analysis.csv")
print(f"\n✓ Missing value analysis saved to {output_dir / 'missing_value_analysis.csv'}")

# 4. Feature Distribution Analysis

Understanding feature distributions is crucial for selecting appropriate preprocessing methods. We'll examine both numerical and categorical features through comprehensive visualizations.


In [ ]:
Features = [
    "Systematic_ID",
    'Chromosome',
    'Start',
    'End',
    'Strand',
    'Abs_distance_from_telomere',
    'Relative_distance_from_telomere',
    'Abs_distance_from_centromere',
    'Relative_distance_from_centromere',
    'Gene_length',
    'GC_content_of_gene',
    'CDS_number',
    'GC_content_of_CDS',
    'Fraction_of_CDS',
    'GC3',
    'Containing_intron',
    'Intron_number',
    'GC_content_of_intron',
    'Total_intron_length',
    'Average_intron_length',
    'Length_of_first_intron',
    'GC_contents_of_first_intron',
    'ENC',
    'Peptide_length',
    'Primary_peptide_length',
    'Primary_candidate',
    'mean_EMM_Nitrogen_Starved_Cell_RNA_Abundance',
    'mean_EMM_Proliferating_Cell_RNA_Abundance',
    'std_EMM_Nitrogen_Starved_Cell_RNA_Abundance',
    'std_EMM_Proliferating_Cell_RNA_Abundance',
    'cv_EMM_Nitrogen_Starved_Cell_RNA_Abundance',
    'cv_EMM_Proliferating_Cell_RNA_Abundance',
    'tAIg',
    'mRNA_half_life_minutes',
    'mRNA_synthesis_rate_per_minute',
    'Mass (kDa)',
    'pI',
    'Charge',
    'Residues',
    'CAI',
    'aromaticity',
    'aliphatic_index',
    'gravy',
    'flexibility',
    'instability_index',
    'aa_percent_Ala',
    'aa_percent_Cys',
    'aa_percent_Asp',
    'aa_percent_Glu',
    'aa_percent_Phe',
    'aa_percent_Gly',
    'aa_percent_His',
    'aa_percent_Ile',
    'aa_percent_Lys',
    'aa_percent_Leu',
    'aa_percent_Met',
    'aa_percent_Asn',
    'aa_percent_Pro',
    'aa_percent_Gln',
    'aa_percent_Arg',
    'aa_percent_Ser',
    'aa_percent_Thr',
    'aa_percent_Val',
    'aa_percent_Trp',
    'aa_percent_Tyr',
    # 'copies_per_cell_EMM_Proliferating_Cell',
    # 'copies_per_cell_EMMN_Quiescent_Cell',
    # 't1/2 (min)',
    'mean_pLDDT',
    'std_pLDDT',
    'cv_pLDDT',
    'PFAM_domain_count',
    'japonicus_ortholog_count',
    'cerevisiae_ortholog_count',
    'human_ortholog_count',
    'paralog_count',
    'evolutionary_rate',
    'mean.phylop',
    'diversity.S',
    'diversity.Pi',
    'diversity.Theta',
    'diversity.Tajima_D',
    'GO_term_richness',
    'PPI_degree',
    'GI_degree',
]

Target = [
    'A',
    'um',
    'lam',
    'DIT_HAP_cluster'
]

organized_all_features_with_target_values = all_features_with_target_values[Features + Target].copy().set_index("Systematic_ID")

In [ ]:
# Identify numerical and categorical columns
numerical_columns = organized_all_features_with_target_values.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = organized_all_features_with_target_values.select_dtypes(include=['object', 'category']).columns.tolist()

# Remove target columns from feature analysis
target_cols = ['A', 'um', 'lam', 'DIT_HAP_cluster']
numerical_features = [col for col in numerical_columns if col not in target_cols]
categorical_features = [col for col in categorical_columns if col not in target_cols]

print(f"Numerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Target variables: {len(target_cols)}")

print(f"\nNumerical features: {numerical_features[:10]}{'...' if len(numerical_features) > 10 else ''}")
print(f"Categorical features: {categorical_features[:10]}{'...' if len(categorical_features) > 10 else ''}")

# Get data for analysis (remove rows with all missing targets)
analysis_data = organized_all_features_with_target_values.copy()
print(f"\nTotal dataset size: {len(analysis_data)}")

# Create a version with complete cases for comparison
complete_cases = analysis_data.dropna()
print(f"Complete cases (no missing values): {len(complete_cases)}")


In [ ]:
# Function to create comprehensive numerical feature histograms
def plot_numerical_distributions(data, features, title_prefix="", max_cols=4):
    """Create histogram plots for numerical features with statistical annotations."""
    
    if not features:
        print("No numerical features to plot.")
        return
    
    n_features = len(features)
    n_cols = min(max_cols, n_features)
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes
    else:
        axes = axes.flatten()
    
    for i, feature in enumerate(features):
        ax = axes[i]
        
        # Get non-null values
        values = data[feature].dropna()
        
        if len(values) == 0:
            ax.text(0.5, 0.5, f'{feature}\n(No data)', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{feature}\n(No data)')
            continue
        
        # Create histogram
        n_bins = min(30, max(10, len(values.unique()) if len(values.unique()) < 50 else 30))
        ax.hist(values, bins=n_bins, alpha=0.8, color=primary_colors[0], edgecolor='white', linewidth=0.5)
        
        # Add statistical annotations
        mean_val = values.mean()
        median_val = values.median()
        std_val = values.std()
        skew_val = skew(values)
        
        # Format statistics text
        stats_text = f'n={len(values):,}\nMean: {mean_val:.2f}\nMedian: {median_val:.2f}\nStd: {std_val:.2f}\nSkew: {skew_val:.2f}'
        
        # Add text box with statistics
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                verticalalignment='top', fontsize=8)
        
        # Add vertical lines for mean and median
        ax.axvline(mean_val, color=primary_colors[1], linestyle='--', alpha=0.7, linewidth=1.5, label='Mean')
        ax.axvline(median_val, color=primary_colors[2], linestyle=':', alpha=0.7, linewidth=1.5, label='Median')
        
        # Formatting
        ax.set_title(f'{feature}', fontsize=10, fontweight='bold')
        ax.set_xlabel('Value', fontsize=9)
        ax.set_ylabel('Frequency', fontsize=9)
        ax.grid(True, alpha=0.3)
        
        # Add legend for mean/median lines
        if i == 0:  # Only add legend to first plot
            ax.legend(loc='upper right', fontsize=8)
    
    # Hide empty subplots
    for i in range(n_features, len(axes)):
        axes[i].set_visible(False)
    
    plt.suptitle(f'{title_prefix}Numerical Feature Distributions', fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    return fig


# Plot feature variable distributions first
print("Creating feature variable distribution plots...")
fig_features = plot_numerical_distributions(analysis_data, numerical_features, "Feature Variables - ")
plt.show()
fig_features.savefig(str(output_dir / "feature_distributions.png"), dpi=300, bbox_inches='tight')
print(f"✓ Feature distributions saved to {output_dir / 'feature_distributions.png'}")
plt.close()

# Plot target variable distributions first
print("Creating target variable distribution plots...")
target_features = ['A', 'um', 'lam', 'DIT_HAP_cluster']
fig_targets = plot_numerical_distributions(analysis_data, target_features, "Target Variables - ")
plt.show()
fig_targets.savefig(str(output_dir / "target_distributions.png"), dpi=300, bbox_inches='tight')
print(f"✓ Target distributions saved to {output_dir / 'target_distributions.png'}")
plt.close()


In [ ]:
# Function to create categorical feature visualizations
def plot_categorical_distributions(data, features, title_prefix="", max_cols=3):
    """Create bar plots for categorical features with frequency counts."""
    
    if not features:
        print("No categorical features to plot.")
        return
    
    n_features = len(features)
    n_cols = min(max_cols, n_features)
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes
    else:
        axes = axes.flatten()
    
    for i, feature in enumerate(features):
        ax = axes[i]
        
        # Get value counts
        value_counts = data[feature].value_counts().sort_index()
        
        if len(value_counts) == 0:
            ax.text(0.5, 0.5, f'{feature}\n(No data)', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{feature}\n(No data)')
            continue
        
        # Create bar plot
        bars = ax.bar(range(len(value_counts)), value_counts.values, 
                     alpha=0.8, color=primary_colors[i % len(primary_colors)], 
                     edgecolor='white', linewidth=0.5)
        
        # Add value labels on bars
        for j, (bar, count) in enumerate(zip(bars, value_counts.values)):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                   f'{count:,}', ha='center', va='bottom', fontsize=8)
        
        # Formatting
        ax.set_title(f'{feature}', fontsize=10, fontweight='bold')
        ax.set_xlabel('Categories', fontsize=9)
        ax.set_ylabel('Frequency', fontsize=9)
        ax.grid(True, axis='y', alpha=0.3)
        
        # Set x-tick labels
        ax.set_xticks(range(len(value_counts)))
        ax.set_xticklabels(value_counts.index, rotation=45, ha='right')
        
        # Add statistics text
        n_categories = len(value_counts)
        n_total = value_counts.sum()
        n_missing = data[feature].isna().sum()
        
        stats_text = f'Categories: {n_categories}\nTotal: {n_total:,}\nMissing: {n_missing:,}'
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                verticalalignment='top', fontsize=8)
    
    # Hide empty subplots
    for i in range(n_features, len(axes)):
        axes[i].set_visible(False)
    
    plt.suptitle(f'{title_prefix}Categorical Feature Distributions', fontsize=14, fontweight='bold', y=0.95)
    plt.tight_layout()
    return fig

# Plot categorical features
print("Creating categorical feature distribution plots...")
if categorical_features:
    fig_categorical = plot_categorical_distributions(analysis_data, categorical_features, "Feature - ")
    plt.show()
    
    # Save categorical distributions
    if fig_categorical is not None:
        fig_categorical.savefig(str(output_dir / "categorical_distributions.png"), dpi=300, bbox_inches='tight')
        print(f"✓ Categorical distributions saved to {output_dir / 'categorical_distributions.png'}")
else:
    print("No categorical features found to visualize.")


In [ ]:
# Special visualization for DIT_HAP_cluster distribution
print("Creating DIT-HAP cluster distribution plot...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Cluster distribution
cluster_counts = analysis_data['DIT_HAP_cluster'].value_counts().sort_index()
if len(cluster_counts) > 0:
    bars1 = ax1.bar(cluster_counts.index, cluster_counts.values, 
                   alpha=0.8, color=primary_colors[3], edgecolor='white', linewidth=0.5)
    
    # Add count labels
    for bar, count in zip(bars1, cluster_counts.values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'{count:,}', ha='center', va='bottom', fontsize=10)
    
    ax1.set_title('DIT-HAP Cluster Distribution', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Cluster', fontsize=10)
    ax1.set_ylabel('Number of Genes', fontsize=10)
    ax1.grid(True, axis='y', alpha=0.3)
    
    # Add statistics
    n_clusters = len(cluster_counts)
    n_assigned = cluster_counts.sum()
    n_missing = analysis_data['DIT_HAP_cluster'].isna().sum()
    
    stats_text = f'Clusters: {n_clusters}\nAssigned: {n_assigned:,}\nUnassigned: {n_missing:,}'
    ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
            verticalalignment='top', fontsize=10)

# Target parameter completeness by cluster
target_params = ['A', 'um', 'lam']
cluster_target_completeness = []

for cluster in sorted(analysis_data['DIT_HAP_cluster'].dropna().unique()):
    cluster_data = analysis_data[analysis_data['DIT_HAP_cluster'] == cluster]
    completeness = {}
    for param in target_params:
        completeness[param] = cluster_data[param].notna().sum() / len(cluster_data)
    cluster_target_completeness.append(completeness)

if cluster_target_completeness:
    cluster_labels = sorted(analysis_data['DIT_HAP_cluster'].dropna().unique())
    x = np.arange(len(cluster_labels))
    width = 0.25
    
    for i, param in enumerate(target_params):
        values = [comp[param] for comp in cluster_target_completeness]
        ax2.bar(x + i*width, values, width, label=param, 
               alpha=0.8, color=primary_colors[i % len(primary_colors)])
    
    ax2.set_title('Target Parameter Completeness by Cluster', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Cluster', fontsize=10)
    ax2.set_ylabel('Proportion Complete', fontsize=10)
    ax2.set_xticks(x + width)
    ax2.set_xticklabels(cluster_labels)
    ax2.legend()
    ax2.grid(True, axis='y', alpha=0.3)
    ax2.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()
# Save cluster visualization
fig.savefig(str(output_dir / "cluster_analysis.png"), dpi=300, bbox_inches='tight')
print(f"✓ Cluster analysis saved to {output_dir / 'cluster_analysis.png'}")
plt.close()


# 5. Feature Engineering and Preprocessing

Based on our exploratory analysis, we'll now apply systematic preprocessing steps including missing value imputation, feature transformation, feature encoding, scaling, and outlier detection.


In [ ]:
# Define feature sets for machine learning
print("=" * 60)
print("FEATURE SELECTION AND ORGANIZATION")
print("=" * 60)

# Define the features we want to use (refined from original notebook)
selected_features_and_transformations = {
    # Basic gene information
    'Systematic_ID': 'set_index',
    'Chromosome': 'OneHotEncoder',
    'Strand': 'binary_encoding',
    
    # Genomic structure features
    'Start': 'PowerTransformer',
    'End': 'PowerTransformer',
    'Abs_distance_from_telomere': 'PowerTransformer',
    'Relative_distance_from_telomere': 'PowerTransformer',
    'Abs_distance_from_centromere': 'PowerTransformer',
    'Relative_distance_from_centromere': 'PowerTransformer',
    'Gene_length': 'PowerTransformer',
    'GC_content_of_gene': 'StandardScaler',
    'CDS_number': 'PowerTransformer',
    'GC_content_of_CDS': 'StandardScaler',
    'Fraction_of_CDS': 'PowerTransformer',
    'GC3': 'StandardScaler',
    'Intron_number': 'PowerTransformer',
    'GC_content_of_intron': 'PowerTransformer',
    'Total_intron_length': 'PowerTransformer',
    'Average_intron_length': 'PowerTransformer',
    'Length_of_first_intron': 'PowerTransformer',
    'GC_contents_of_first_intron': 'PowerTransformer',
    'ENC': 'PowerTransformer',
    'Peptide_length': 'PowerTransformer',
    'Primary_peptide_length': 'PowerTransformer',
    'mean_EMM_Nitrogen_Starved_Cell_RNA_Abundance': 'PowerTransformer',
    'mean_EMM_Proliferating_Cell_RNA_Abundance': 'PowerTransformer',
    'std_EMM_Nitrogen_Starved_Cell_RNA_Abundance': 'PowerTransformer',
    'std_EMM_Proliferating_Cell_RNA_Abundance': 'PowerTransformer',
    'cv_EMM_Nitrogen_Starved_Cell_RNA_Abundance': 'PowerTransformer',
    'cv_EMM_Proliferating_Cell_RNA_Abundance': 'PowerTransformer',
    'tAIg': 'StandardScaler',
    'mRNA_half_life_minutes': 'PowerTransformer',
    'mRNA_synthesis_rate_per_minute': 'PowerTransformer',
    'Mass (kDa)': 'PowerTransformer',
    'pI': 'StandardScaler',
    'Charge': 'StandardScaler',
    'Residues': 'PowerTransformer',
    'CAI': 'StandardScaler',
    'aromaticity': 'StandardScaler',
    'aliphatic_index': 'StandardScaler',
    'gravy': 'StandardScaler',
    'flexibility': 'StandardScaler',
    'instability_index': 'StandardScaler',
    'aa_percent_Ala': 'StandardScaler',
    'aa_percent_Cys': 'StandardScaler',
    'aa_percent_Asp': 'StandardScaler',
    'aa_percent_Glu': 'StandardScaler',
    'aa_percent_Phe': 'StandardScaler',
    'aa_percent_Gly': 'StandardScaler',
    'aa_percent_His': 'StandardScaler',
    'aa_percent_Ile': 'StandardScaler',
    'aa_percent_Lys': 'StandardScaler',
    'aa_percent_Leu': 'StandardScaler',
    'aa_percent_Met': 'StandardScaler',
    'aa_percent_Asn': 'StandardScaler',
    'aa_percent_Pro': 'StandardScaler',
    'aa_percent_Gln': 'StandardScaler',
    'aa_percent_Arg': 'StandardScaler',
    'aa_percent_Ser': 'StandardScaler',
    'aa_percent_Thr': 'StandardScaler',
    'aa_percent_Val': 'StandardScaler',
    'aa_percent_Trp': 'StandardScaler',
    'aa_percent_Tyr': 'StandardScaler',
    # 'copies_per_cell_EMM_Proliferating_Cell': 'PowerTransformer',
    # 'copies_per_cell_EMMN_Quiescent_Cell': 'PowerTransformer',
    # 't1/2 (min)': 'PowerTransformer',
    'mean_pLDDT': 'StandardScaler',
    'std_pLDDT': 'StandardScaler',
    'cv_pLDDT': 'StandardScaler',
    'PFAM_domain_count': 'PowerTransformer',
    'japonicus_ortholog_count': 'PowerTransformer',
    'cerevisiae_ortholog_count': 'PowerTransformer',
    'human_ortholog_count': 'PowerTransformer',
    'paralog_count': 'PowerTransformer',
    'evolutionary_rate': 'StandardScaler',
    'mean.phylop': 'StandardScaler',
    'diversity.S': 'PowerTransformer',
    'diversity.Pi': 'PowerTransformer',
    'diversity.Theta': 'StandardScaler',
    'diversity.Tajima_D': 'StandardScaler',
    'GO_term_richness': 'PowerTransformer',
    'PPI_degree': 'PowerTransformer',
    'GI_degree': 'PowerTransformer',
}
selected_features = list(selected_features_and_transformations.keys())

# Target variables
target_features = ['A', 'um', 'lam', 'DIT_HAP_cluster']

# Check which features are available in our dataset
available_features = [f for f in selected_features if f in all_features_with_target_values.columns]
missing_features = [f for f in selected_features if f not in all_features_with_target_values.columns]

print(f"Selected features: {len(selected_features)}")
print(f"Available features: {len(available_features)}")
print(f"Missing features: {len(missing_features)}")

if missing_features:
    print(f"\nMissing features: {missing_features}")

# Create working dataset with selected features
working_data = all_features_with_target_values[available_features + target_features].copy().dropna(axis=0, how='any').set_index("Systematic_ID")
working_data_um_gt_p35 = all_features_with_target_values.query("um > 0.35")[available_features + target_features].copy().dropna(axis=0, how='any').set_index("Systematic_ID")
working_data_um_le_p35 = all_features_with_target_values.query("um <= 0.35")[available_features + target_features].copy().dropna(axis=0, how='any').set_index("Systematic_ID")
print(f"\nWorking dataset shape: {working_data.shape}")
print(f"\nWorking dataset shape: {working_data_um_gt_p35.shape}")
print(f"\nWorking dataset shape: {working_data_um_le_p35.shape}")

# Display missing value summary for selected features
for data in [working_data, working_data_um_gt_p35, working_data_um_gt_p35]:
    missing_in_selected = data.isna().sum()
    missing_in_selected_sorted = missing_in_selected.sort_values(ascending=False)
    missing_in_selected_filtered = missing_in_selected_sorted[missing_in_selected_sorted > 0]

    print(f"\nMissing values in selected features:")
    if len(missing_in_selected_filtered) > 0:
        display(missing_in_selected_filtered.head(10))
    else:
        print("No missing values in selected features!")


In [ ]:
def transform_data(data, selected_features_and_transformations, target_features):
    transformed_data = pd.DataFrame()
    for feature, transformation in selected_features_and_transformations.items():
        print(f"\nFeature: {feature}")
        print(f"Transformation: {transformation}")
        if transformation == 'set_index':
            transformed_data[feature] = data.index.tolist()
            transformed_data.set_index(feature, inplace=True)
        elif transformation == 'OneHotEncoder':
            values = pd.get_dummies(data[feature], prefix=feature, dummy_na=True).astype(int)
            transformed_data = pd.concat([transformed_data, values], axis=1)
        elif transformation == 'binary_encoding':
            # Convert '+' to 1 and '-' to 0
            transformed_data[feature] = data[feature].map({'+': 1, '-': 0})
        elif transformation == 'StandardScaler':
            transformed_data[feature] = StandardScaler().fit_transform(data[feature].values.reshape(-1, 1))
        elif transformation == 'np.log1p':
            transformed_data[feature] = np.log1p(data[feature])
        elif transformation == 'PowerTransformer':
            transformed_data[feature] = PowerTransformer().fit_transform(data[feature].values.reshape(-1, 1))
        elif transformation == 'No processing':
            transformed_data[feature] = data[feature]
        else:
            transformed_data[feature] = data[feature]

    transformed_data[target_features] = data[target_features]
    return transformed_data

datas = {
    "all": working_data,
    "um_gt_p35": working_data_um_gt_p35,
    "um_le_p35": working_data_um_le_p35
}

transformed_datas = {}
for des, data in datas.items():
    transformed_datas[des] = transform_data(data, selected_features_and_transformations, target_features)

In [ ]:
for des, transformed_data in transformed_datas.items():
    print("Creating transformedfeature variable distribution plots...")
    fig_features = plot_numerical_distributions(transformed_data, transformed_data.columns.tolist(), "Transformed Feature Variables - ")
    plt.show()
    fig_features.savefig(str(output_dir / f"{des}_transformed_feature_distributions.png"), dpi=300, bbox_inches='tight')
    print(f"✓ Transformed feature distributions saved to {output_dir / f'{des}_transformed_feature_distributions.png'}")
    plt.close()

# 6. Comprehensive Feature-Target Relationship Visualization

We'll create scatter plots and box plots for all combinations between features and target variables to understand relationships and identify potential predictors.

In [ ]:
# Function to create scatter plots for numerical vs numerical relationships
def create_scatter_plots(data, features, targets, title_prefix="", max_plots_per_figure=12):
    """Create scatter plots between numerical features and numerical targets."""
    
    numerical_targets = [t for t in targets if t in ['A', 'um', 'lam']]
    if not numerical_targets or not features:
        return []
    
    figures = []
    n_combinations = len(features) * len(numerical_targets)
    n_figures = (n_combinations + max_plots_per_figure - 1) // max_plots_per_figure
    
    for fig_idx in range(n_figures):
        start_idx = fig_idx * max_plots_per_figure
        end_idx = min(start_idx + max_plots_per_figure, n_combinations)
        n_plots_this_fig = end_idx - start_idx
        
        n_cols = min(3, n_plots_this_fig)
        n_rows = (n_plots_this_fig + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))
        if n_plots_this_fig == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes if hasattr(axes, '__len__') else [axes]
        else:
            axes = axes.flatten()
        
        plot_idx = 0
        combination_idx = 0
        
        for feature in features:
            for target in numerical_targets:
                if combination_idx < start_idx:
                    combination_idx += 1
                    continue
                if combination_idx >= end_idx:
                    break
                    
                ax = axes[plot_idx]
                
                # Get clean data
                clean_data = data[[feature, target]].dropna()
                
                if len(clean_data) < 10:
                    ax.text(0.5, 0.5, f'{feature} vs {target}\nInsufficient data', 
                           ha='center', va='center', transform=ax.transAxes)
                else:
                    # Create scatter plot
                    x = clean_data[feature]
                    y = clean_data[target]
                    try:
                        xy = np.vstack([x, y])
                        z = gaussian_kde(xy)(xy)
                        ax.scatter(clean_data[feature], clean_data[target], s=20, c=z, edgecolor="none")
                    except Exception as e:
                        print(f"Error plotting {feature} vs {target}: {e}")
                        ax.scatter(clean_data[feature], clean_data[target], s=20, color=primary_colors[0], edgecolor="none")
                    
                    # Calculate and display correlation
                    corr, p_val = pearsonr(clean_data[feature], clean_data[target])
                    
                    # Add trend line if correlation is significant
                    if p_val < 0.05:
                        z = np.polyfit(clean_data[feature], clean_data[target], 1)
                        p = np.poly1d(z)
                        ax.plot(clean_data[feature], p(clean_data[feature]), 
                               color='red', linestyle='--', alpha=0.8, linewidth=1)
                    
                    # Add correlation text
                    ax.text(0.05, 0.95, f'r={corr:.3f}\np={p_val:.3e}', 
                           transform=ax.transAxes, 
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                           verticalalignment='top', fontsize=8)
                
                ax.set_xlabel(feature, fontsize=9)
                ax.set_ylabel(target, fontsize=9)
                ax.set_title(f'{feature} vs {target}', fontsize=10, fontweight='bold')
                ax.grid(True, alpha=0.3)
                
                plot_idx += 1
                combination_idx += 1
                
            if combination_idx >= end_idx:
                break
        
        # Hide unused subplots
        for i in range(plot_idx, len(axes)):
            axes[i].set_visible(False)
        
        plt.suptitle(f'{title_prefix}Feature-Target Relationships (Part {fig_idx+1})', 
                    fontsize=14, fontweight='bold', y=0.98)
        plt.tight_layout()
        figures.append(fig)
    
    return figures

for des, transformed_data in transformed_datas.items():
    data = datas[des]
    working_feature_columns = [col for col in data.select_dtypes(include=['number']).columns if col not in target_features]
    transformed_feature_columns = [col for col in transformed_data.columns if col not in target_features]

    print("\nCreating scatter plots for numerical features vs targets...")
    scatter_figs = create_scatter_plots(data, working_feature_columns, target_features, "Numerical ")
    with PdfPages(output_dir / f"{des}_scatter_plots_numerical.pdf") as pdf:
        for fig in scatter_figs:
            pdf.savefig(fig, dpi=300, bbox_inches='tight')
            plt.close(fig)

    print("\nCreating scatter plots for log-transformed features vs targets...")
    log_scatter_figs = create_scatter_plots(transformed_data, transformed_feature_columns, target_features, "Transformed ")
    with PdfPages(output_dir / f"{des}_scatter_plots_transformed.pdf") as pdf:
        for fig in log_scatter_figs:
            pdf.savefig(fig, dpi=300, bbox_inches='tight')
            plt.close(fig)


In [ ]:
# Function to create box plots for categorical vs numerical relationships
def create_box_plots(data, categorical_features, numerical_targets, title_prefix=""):
    """Create box plots between categorical features and numerical targets."""
    
    if not categorical_features or not numerical_targets:
        return []
    
    figures = []
    n_combinations = len(categorical_features) * len(numerical_targets)
    
    # Create one figure for all combinations
    n_cols = min(3, n_combinations)
    n_rows = (n_combinations + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    if n_combinations == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes if hasattr(axes, '__len__') else [axes]
    else:
        axes = axes.flatten()
    
    plot_idx = 0
    
    for cat_feature in categorical_features:
        for num_target in numerical_targets:
            ax = axes[plot_idx]
            
            # Get clean data
            clean_data = data[[cat_feature, num_target]].dropna()
            
            if len(clean_data) < 10:
                ax.text(0.5, 0.5, f'{cat_feature} vs {num_target}\nInsufficient data', 
                       ha='center', va='center', transform=ax.transAxes)
            else:
                # Create box plot
                categories = sorted(clean_data[cat_feature].unique())
                box_data = [clean_data[clean_data[cat_feature] == cat][num_target].values 
                           for cat in categories]
                
                bp = ax.boxplot(box_data, labels=categories, patch_artist=True)
                
                # Color the boxes
                for patch, color in zip(bp['boxes'], primary_colors):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.7)
                
                # Add sample sizes
                for i, cat in enumerate(categories):
                    n_samples = len(clean_data[clean_data[cat_feature] == cat])
                    ax.text(i+1, ax.get_ylim()[1]*0.95, f'n={n_samples}', 
                           ha='center', va='top', fontsize=8)
                
                # Perform statistical test if there are multiple categories
                if len(categories) > 1:
                    from scipy.stats import kruskal
                    try:
                        stat, p_val = kruskal(*box_data)
                        ax.text(0.02, 0.98, f'Kruskal-Wallis\np={p_val:.3e}', 
                               transform=ax.transAxes,
                               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                               verticalalignment='top', fontsize=8)
                    except:
                        pass
            
            ax.set_xlabel(cat_feature, fontsize=10)
            ax.set_ylabel(num_target, fontsize=10)
            ax.set_title(f'{num_target} by {cat_feature}', fontsize=11, fontweight='bold')
            ax.grid(True, axis='y', alpha=0.3)
            
            plot_idx += 1
    
    # Hide unused subplots
    for i in range(plot_idx, len(axes)):
        axes[i].set_visible(False)
    
    plt.suptitle(f'{title_prefix}Categorical Feature vs Target Relationships', 
                fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    figures.append(fig)
    
    return figures

for des, transformed_data in transformed_datas.items():
    data = datas[des]
    # Create box plots for categorical features vs numerical targets
    numerical_targets_only = [t for t in target_features if t in ['A', 'um', 'lam']]
    selected_categorical_features = [f for f in data.columns if f in data.select_dtypes(include=['object']).columns]

    print("\nCreating box plots for categorical features vs numerical targets...")
    box_figs = create_box_plots(data, selected_categorical_features, numerical_targets_only, "")
    with PdfPages(output_dir / f"{des}_box_plots_categorical_vs_targets.pdf") as pdf:
        for fig in box_figs:
            pdf.savefig(fig, dpi=300, bbox_inches='tight')
            plt.close(fig)

    print(f"✓ Box plots saved to {output_dir / f'{des}_box_plots_categorical_vs_targets.pdf'}")


In [ ]:
# Special analysis: Features vs DIT_HAP_cluster (categorical target)
for des, transformed_data in transformed_datas.items():
    print("\nCreating visualizations for features vs DIT_HAP_cluster...")

    # Box plots for numerical features vs clusters
    cluster_data = transformed_data[transformed_data['DIT_HAP_cluster'].notna()]
    if len(cluster_data) > 0:
        
        features_for_cluster = [ c for c in transformed_data.columns if c != 'DIT_HAP_cluster']
        
        n_features = len(features_for_cluster)
        n_cols = 3
        n_rows = (n_features + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 4*n_rows))
        if n_features == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes if hasattr(axes, '__len__') else [axes]
        else:
            axes = axes.flatten()
        
        for i, feature in enumerate(features_for_cluster):
            ax = axes[i]
            
            # Get clean data for this feature
            feature_cluster_data = cluster_data[[feature, 'DIT_HAP_cluster']].dropna()
            
            if len(feature_cluster_data) > 10:
                # Create box plot
                clusters = sorted(feature_cluster_data['DIT_HAP_cluster'].unique())
                box_data = [feature_cluster_data[feature_cluster_data['DIT_HAP_cluster'] == cluster][feature].values 
                        for cluster in clusters]
                
                bp = ax.boxplot(box_data, labels=[f'C{int(c)}' for c in clusters], patch_artist=True)
                
                # Color boxes by cluster
                for j, patch in enumerate(bp['boxes']):
                    patch.set_facecolor(primary_colors[j % len(primary_colors)])
                    patch.set_alpha(0.7)
                
                # Add sample sizes
                for j, cluster in enumerate(clusters):
                    n_samples = len(feature_cluster_data[feature_cluster_data['DIT_HAP_cluster'] == cluster])
                    ax.text(j+1, ax.get_ylim()[1]*0.95, f'n={n_samples}', 
                        ha='center', va='top', fontsize=8)
                
                # Statistical test
                if len(clusters) > 1:
                    from scipy.stats import kruskal
                    try:
                        stat, p_val = kruskal(*box_data)
                        ax.text(0.02, 0.98, f'p={p_val:.3e}', transform=ax.transAxes,
                            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                            verticalalignment='top', fontsize=8)
                    except:
                        pass
            
            ax.set_xlabel('DIT_HAP_cluster', fontsize=10)
            ax.set_ylabel(feature, fontsize=10)
            ax.set_title(f'{feature} by Cluster', fontsize=11, fontweight='bold')
            ax.grid(True, axis='y', alpha=0.3)
        
        # Hide unused subplots
        for i in range(len(features_for_cluster), len(axes)):
            axes[i].set_visible(False)
        
        plt.suptitle('Feature Distributions Across DIT-HAP Clusters', fontsize=14, fontweight='bold', y=0.98)
        plt.tight_layout()
        plt.show()
        # Save cluster analysis
        fig.savefig(str(output_dir / f"{des}_features_vs_clusters.pdf"), dpi=300, bbox_inches='tight')
        print(f"✓ Cluster analysis saved to {output_dir / f'{des}_features_vs_clusters.pdf'}")

    else:
        print("⚠ No cluster data available for visualization")


In [ ]:
# Select key features for visualization
key_features = [
    "mean_EMM_Proliferating_Cell_RNA_Abundance",
    "mRNA_half_life_minutes",
    "mRNA_synthesis_rate_per_minute",
    "copies_per_cell_EMM_Proliferating_Cell",
    "t1/2 (min)",
    "evolutionary_rate"
]

In [ ]:
alt.data_transformers.enable("vegafusion")
alt.Chart(all_features_with_target_values.dropna(subset=["DIT_HAP_cluster"]).query("DIT_HAP_cluster < 9")).mark_boxplot(outliers=False).encode(
    x=alt.X(alt.repeat("column"), type="quantitative"),
    y=alt.Y("DIT_HAP_cluster:O", title="DIT-HAP Cluster"),
).repeat(
    column=key_features
).resolve_scale(
    x='independent'
)

In [ ]:
# Create boxplots for key features using matplotlib
print("\nCreating boxplots for key features across DIT-HAP clusters...")

# Filter data for clusters < 5 and remove NaN values
plot_data = all_features_with_target_values.dropna(subset=["DIT_HAP_cluster"])
plot_data = plot_data[plot_data["DIT_HAP_cluster"] < 7]

if not plot_data.empty:
    # Create subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 6))
    axes = axes.flatten()
    
    # Define colors for each cluster
    cluster_colors = [
            "#dd8369",
            "#6b99df",
            "#98a64e",
            "#64af6d",
            "#a78bd9",
            "#d57fbd",
            "#c4954b",
            "#4bb29c",
            "#e0788f",
            "#4aadce",
        ]
    
    for i, feature in enumerate(key_features):
        ax = axes[i]
        
        # Get data for each cluster
        cluster_data = []
        cluster_labels = []
        colors = []
        
        # Sort clusters in reverse order (4 at top, 1 at bottom)
        for cluster in sorted(plot_data["DIT_HAP_cluster"].unique(), reverse=True):
            feature_values = plot_data[plot_data["DIT_HAP_cluster"] == cluster][feature].dropna()
            if len(feature_values) > 0:
                cluster_data.append(feature_values)
                cluster_labels.append(f'{int(cluster)}')
                colors.append(cluster_colors[int(cluster) % len(cluster_colors)])
        
        if cluster_data:
            # Create horizontal boxplot (cluster on y-axis, feature on x-axis)
            bp = ax.boxplot(cluster_data, labels=cluster_labels, patch_artist=True,
                           vert=False, showfliers=False,
                           boxprops=dict(linewidth=1.5),
                           medianprops=dict(color='black', linewidth=2),
                           whiskerprops=dict(linewidth=1.5),
                           capprops=dict(linewidth=1.5))
            
            # Color the boxes
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
        
        # Formatting
        if i == 0:
            ax.set_ylabel('DIT-HAP Cluster', fontsize=14, fontweight='bold')
        ax.set_yticklabels(cluster_labels, fontsize=14, fontweight='bold')
        ax.set_xlabel(feature.replace('_', ' '), fontsize=14, fontweight='bold')
        ax.set_title(f'{feature.replace("_", " ")} by Cluster', 
                    fontsize=16, fontweight='bold', pad=10)
        ax.grid(True, axis='x', alpha=0.3, linestyle='--')
        ax.tick_params(axis='both', which='major', labelsize=14)

        if i != 5:
            ax.set_xscale('log')
    
    # Overall formatting
    # plt.suptitle('Feature Distributions Across DIT-HAP Clusters', 
    #             fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout(h_pad=5)
    plt.subplots_adjust(top=0.93)
    plt.show()
    
    # Save the plot
    fig.savefig(str(output_dir / "key_features_boxplots.pdf"), dpi=300, bbox_inches='tight')
    print(f"✓ Key features boxplots saved to {output_dir / 'key_features_boxplots.pdf'}")
    plt.close()

else:
    print("⚠ No data available for boxplot visualization")
 

In [ ]:
# Create boxplots for key features using matplotlib
print("\nCreating boxplots for key features across DIT-HAP clusters...")

# Filter data for clusters < 5 and remove NaN values
plot_data = all_features_with_target_values.dropna(subset=["DIT_HAP_cluster"])
plot_data = plot_data[plot_data["DIT_HAP_cluster"] < 7]

if not plot_data.empty:
    # Create subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 6))
    axes = axes.flatten()
    
    # Define colors for each cluster
    cluster_colors = [
            "#dd8369",
            "#6b99df",
            "#98a64e",
            "#64af6d",
            "#a78bd9",
            "#d57fbd",
            "#c4954b",
            "#4bb29c",
            "#e0788f",
            "#4aadce",
        ]
    
    for i, feature in enumerate(key_features):
        ax = axes[i]
        
        # Get data for each cluster
        cluster_data = []
        cluster_labels = []
        colors = []
        
        # Sort clusters in reverse order (4 at top, 1 at bottom)
        for cluster in sorted(plot_data["DIT_HAP_cluster"].unique(), reverse=True):
            feature_values = plot_data[plot_data["DIT_HAP_cluster"] == cluster][feature].dropna()
            if i != 5:
                feature_values = np.log1p(feature_values)
            if len(feature_values) > 0:
                cluster_data.append(feature_values)
                cluster_labels.append(f'{int(cluster)}')
                colors.append(cluster_colors[int(cluster) % len(cluster_colors)])
        
        if cluster_data:
            # Create horizontal boxplot (cluster on y-axis, feature on x-axis)
            bp = ax.boxplot(cluster_data, labels=cluster_labels, patch_artist=True,
                           vert=False, showfliers=False,
                           boxprops=dict(linewidth=1.5),
                           medianprops=dict(color='black', linewidth=2),
                           whiskerprops=dict(linewidth=1.5),
                           capprops=dict(linewidth=1.5))
            
            # Color the boxes
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
        
        # Formatting
        if i == 0:
            ax.set_ylabel('DIT-HAP Cluster', fontsize=14, fontweight='bold')
        ax.set_yticklabels(cluster_labels, fontsize=14, fontweight='bold')
        ax.set_xlabel(feature.replace('_', ' '), fontsize=14, fontweight='bold')
        ax.set_title(f'{feature.replace("_", " ")} by Cluster', 
                    fontsize=16, fontweight='bold', pad=10)
        ax.grid(True, axis='x', alpha=0.3, linestyle='--')
        ax.tick_params(axis='both', which='major', labelsize=14)

        # if i != 5:
        #     ax.set_xscale('log')
    
    # Overall formatting
    # plt.suptitle('Feature Distributions Across DIT-HAP Clusters', 
    #             fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout(h_pad=5)
    plt.subplots_adjust(top=0.93)
    plt.show()
    
    # Save the plot
    fig.savefig(str(output_dir / "key_features_boxplots.pdf"), dpi=300, bbox_inches='tight')
    print(f"✓ Key features boxplots saved to {output_dir / 'key_features_boxplots.pdf'}")
    plt.close()

else:
    print("⚠ No data available for boxplot visualization")
 

In [ ]:
# Create correlation heatmap for key numerical features and targets
print("\nCreating correlation heatmap...")
for des, transformed_data in transformed_datas.items():
    # Select features for correlation analysis
    corr_features = transformed_data.columns.tolist()
    corr_data = transformed_data[corr_features].corr()

    # Create heatmap
    fig, ax = plt.subplots(figsize=(30, 24))
    mask = np.triu(np.ones_like(corr_data, dtype=bool))  # Hide upper triangle
    sns.heatmap(corr_data, mask=mask, annot=True, cmap='RdBu_r', center=0, 
                square=True, linewidths=0.5, cbar_kws={"shrink": .8}, fmt='.2f')

    plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    # Save correlation heatmap
    fig.savefig(str(output_dir / f"{des}_correlation_heatmap.pdf"), dpi=300, bbox_inches='tight')
    print(f"✓ Correlation heatmap saved to {output_dir / f'{des}_correlation_heatmap.pdf'}")
    plt.close()

# 7.Output the data

In [ ]:
for des, transformed_data in transformed_datas.items():
    feature_columns = [col for col in transformed_data.columns if col not in target_features]
    transformed_data.to_csv(output_dir / f"{des}_transformed_features_and_targets.csv", index=True, float_format="%.3f")
    transformed_data[feature_columns].to_csv(output_dir / f"{des}_transformed_features.csv", index=True, float_format="%.3f")
    transformed_data[target_features].to_csv(output_dir / f"{des}_transformed_targets.csv", index=True, float_format="%.3f")

In [ ]:
non_WT_cluster_data = transformed_datas["all"].query('DIT_HAP_cluster != 9').copy()
feature_columns = [col for col in non_WT_cluster_data.columns if col not in target_features]
non_WT_cluster_data.to_csv(output_dir / "nonWT_transformed_features_and_targets.csv", index=True, float_format="%.3f")
non_WT_cluster_data[feature_columns].to_csv(output_dir / "nonWT_transformed_features.csv", index=True, float_format="%.3f")
non_WT_cluster_data[target_features].to_csv(output_dir / "nonWT_transformed_targets.csv", index=True, float_format="%.3f")